# Regression Models — Predicting Delay Minutes

In addition to binary classification (`DepDel15`), we predict the **continuous departure delay in minutes** (`DepDelay`).

## Target
- `DepDelay`: departure delay in minutes (negative = early, 0 = on-time, positive = delayed)

## Models
1. **Linear Regression** — baseline
2. **Ridge Regression** — L2 regularized
3. **LightGBM Regressor** — gradient boosted trees

## Metrics
- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)
- R² (Coefficient of Determination)

## Split
Same temporal split: train on months 1–10, test on months 11–12.

## Step 0: Setup

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data
ensure_project_data()

In [ ]:
import os, re, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor

from project_data import resolve_project_root

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
INTEGRATED_DIR = DATA_ROOT / "processed" / "integrated"
REPORT_DIR = DATA_ROOT / "reports" / "modeling"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INTEGRATED_DIR / "features_2024.parquet")
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

## Step 1: Target variable — `DepDelay`

In [ ]:
target = df["DepDelay"].dropna()
print(f"DepDelay stats (n={len(target):,}):")
print(target.describe().to_string())
print(f"\nSkewness: {target.skew():.2f}")
print(f"% negative (early): {(target < 0).mean()*100:.1f}%")
print(f"% zero (on-time):   {(target == 0).mean()*100:.1f}%")
print(f"% positive (late):  {(target > 0).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram — clipped to [-30, 120] for readability
ax = axes[0]
clipped = target.clip(-30, 120)
ax.hist(clipped, bins=100, color="#2563EB", edgecolor="white", alpha=0.8)
ax.axvline(0, color="#DC2626", linestyle="--", linewidth=1.5, label="On-time (0 min)")
ax.axvline(15, color="#F59E0B", linestyle="--", linewidth=1.5, label="DepDel15 threshold (15 min)")
ax.axvline(target.median(), color="#10B981", linestyle="--", linewidth=1.5,
           label=f"Median ({target.median():.0f} min)")
ax.set_xlabel("Departure Delay (minutes)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of DepDelay (clipped to [-30, 120])", fontweight="bold")
ax.legend(fontsize=9)

# Box plot by month
ax = axes[1]
monthly = df[["month", "DepDelay"]].dropna()
monthly_clipped = monthly.copy()
monthly_clipped["DepDelay"] = monthly_clipped["DepDelay"].clip(-30, 120)
monthly_clipped.boxplot(column="DepDelay", by="month", ax=ax,
                         flierprops=dict(marker='.', markersize=1, alpha=0.1))
ax.set_xlabel("Month")
ax.set_ylabel("Departure Delay (minutes)")
ax.set_title("DepDelay by Month", fontweight="bold")
plt.suptitle("")  # remove auto title

plt.tight_layout()
plt.savefig(REPORT_DIR / "regression_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 2: Feature matrix

In [ ]:
TARGET = "DepDelay"

# Same feature set as classification, minus DepDel15-related leakage
NUMERIC_FEATURES = [
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "dep_hour", "month", "day_of_week", "is_weekend",
    "is_holiday", "holiday_proximity",
    "is_origin_hub", "is_dest_hub",
    "airline_delay_rate_7d", "origin_delay_rate_7d", "route_delay_rate_7d",
    "origin_daily_flights",
    "prev_flight_arr_delay", "tail_leg_today",
    "origin_hourly_flights",
    "origin_air_temp", "origin_dew_point", "origin_sea_level_pres",
    "origin_wind_dir", "origin_wind_speed", "origin_sky_cover",
    "origin_precip_1h", "origin_precip_6h",
    "origin_weather_severity",
    "origin_freezing_rain", "origin_wind_rain", "origin_fog_risk",
    "dest_air_temp", "dest_dew_point", "dest_sea_level_pres",
    "dest_wind_dir", "dest_wind_speed", "dest_sky_cover",
    "dest_precip_1h", "dest_precip_6h",
    "dest_weather_severity",
    "dest_freezing_rain", "dest_wind_rain", "dest_fog_risk",
    "worst_precip", "worst_wind",
    "origin_is_rain", "origin_high_wind", "origin_freezing", "origin_low_vis",
    "dest_is_rain", "dest_high_wind", "dest_freezing", "dest_low_vis",
]

def top_n_encode(series, n=30):
    top = series.value_counts().nlargest(n).index
    return series.where(series.isin(top), "OTHER")

df["Origin_enc"] = top_n_encode(df["Origin"].astype(str), 30)
df["Dest_enc"] = top_n_encode(df["Dest"].astype(str), 30)

cat_encode_cols = ["Reporting_Airline", "Origin_enc", "Dest_enc", "time_block", "distance_bin"]
dummies = pd.get_dummies(df[cat_encode_cols].astype(str), drop_first=True, dtype=int)

X = pd.concat([df[NUMERIC_FEATURES], dummies], axis=1)
X = X.fillna(X.median())
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype(np.float64)
X.columns = [re.sub(r'[\[\]<]', '_', c) for c in X.columns]

# Drop rows where target is NaN
valid_mask = df[TARGET].notna()
X = X.loc[valid_mask]
y_all = df.loc[valid_mask, TARGET].astype(np.float64)

print(f"Features: {X.shape[1]}")
print(f"Valid rows: {len(X):,} (dropped {(~valid_mask).sum():,} with NaN target)")

## Step 3: Temporal split + subsample

In [ ]:
month_col = df.loc[valid_mask, "month"]
train_mask = month_col <= 10
test_mask = month_col >= 11

y_full_train = y_all[train_mask]
y_test = y_all[test_mask]

# Stratified subsample for training (500k)
TRAIN_SAMPLE = 500_000
rng = np.random.RandomState(42)
train_idx = X.index[train_mask]

if len(train_idx) > TRAIN_SAMPLE:
    sampled_idx = rng.choice(train_idx, TRAIN_SAMPLE, replace=False)
    train_idx = pd.Index(sampled_idx)

X_train = X.loc[train_idx].copy()
X_test = X.loc[test_mask].copy()
y_train = y_all.loc[train_idx]

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train DepDelay — mean: {y_train.mean():.1f} min, median: {y_train.median():.1f} min")
print(f"Test  DepDelay — mean: {y_test.mean():.1f} min, median: {y_test.median():.1f} min")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaling complete.")

## Step 4: Naive baseline

Always predict the training set mean delay. This gives us a floor for comparison.

In [ ]:
y_pred_naive = np.full(len(y_test), y_train.mean())

rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))
mae_naive = mean_absolute_error(y_test, y_pred_naive)
r2_naive = r2_score(y_test, y_pred_naive)

print(f"Naive Baseline (predict mean = {y_train.mean():.2f} min)")
print(f"  RMSE: {rmse_naive:.2f} min")
print(f"  MAE:  {mae_naive:.2f} min")
print(f"  R²:   {r2_naive:.4f}")

## Step 5: Linear Regression

In [ ]:
print("Training Linear Regression...")
t0 = time.time()
lr = LinearRegression(n_jobs=-1)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
elapsed = time.time() - t0

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Done in {elapsed:.1f}s")
print(f"Linear Regression:")
print(f"  RMSE: {rmse_lr:.2f} min")
print(f"  MAE:  {mae_lr:.2f} min")
print(f"  R²:   {r2_lr:.4f}")

## Step 6: Ridge Regression

In [ ]:
print("Training Ridge Regression...")
t0 = time.time()
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)
elapsed = time.time() - t0

rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Done in {elapsed:.1f}s")
print(f"Ridge Regression (α=1.0):")
print(f"  RMSE: {rmse_ridge:.2f} min")
print(f"  MAE:  {mae_ridge:.2f} min")
print(f"  R²:   {r2_ridge:.4f}")

## Step 7: LightGBM Regressor

In [ ]:
print("Training LightGBM Regressor...")
t0 = time.time()
lgbm_reg = LGBMRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgbm_reg.fit(X_train, y_train)  # tree models don't need scaling

y_pred_lgbm = lgbm_reg.predict(X_test)
elapsed = time.time() - t0

rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
r2_lgbm = r2_score(y_test, y_pred_lgbm)

print(f"Done in {elapsed:.1f}s")
print(f"LightGBM Regressor:")
print(f"  RMSE: {rmse_lgbm:.2f} min")
print(f"  MAE:  {mae_lgbm:.2f} min")
print(f"  R²:   {r2_lgbm:.4f}")

## Step 8: Model comparison

In [ ]:
results = pd.DataFrame([
    {"Model": "Naive (predict mean)", "RMSE": rmse_naive, "MAE": mae_naive, "R²": r2_naive},
    {"Model": "Linear Regression", "RMSE": rmse_lr, "MAE": mae_lr, "R²": r2_lr},
    {"Model": "Ridge Regression", "RMSE": rmse_ridge, "MAE": mae_ridge, "R²": r2_ridge},
    {"Model": "LightGBM Regressor", "RMSE": rmse_lgbm, "MAE": mae_lgbm, "R²": r2_lgbm},
])

print("=" * 65)
print("REGRESSION MODEL COMPARISON — TEST SET (Nov–Dec 2024)")
print("=" * 65)
print(results.to_string(index=False, float_format="{:.4f}".format))
print("=" * 65)

results.to_csv(REPORT_DIR / "regression_model_comparison.csv", index=False)

## Step 9: Residual analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models_data = [
    ("Linear Regression", y_pred_lr),
    ("Ridge Regression", y_pred_ridge),
    ("LightGBM Regressor", y_pred_lgbm),
]

# Panel A: Predicted vs Actual (LightGBM — best model)
ax = axes[0, 0]
sample_idx = rng.choice(len(y_test), min(20000, len(y_test)), replace=False)
y_test_arr = y_test.values
ax.scatter(y_test_arr[sample_idx], y_pred_lgbm[sample_idx],
           alpha=0.05, s=2, color="#2563EB")
lims = [-30, 150]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual DepDelay (min)")
ax.set_ylabel("Predicted DepDelay (min)")
ax.set_title("LightGBM: Predicted vs Actual", fontweight="bold")
ax.legend()

# Panel B: Residual distribution (LightGBM)
ax = axes[0, 1]
residuals = y_test_arr - y_pred_lgbm
res_clipped = np.clip(residuals, -100, 100)
ax.hist(res_clipped, bins=100, color="#2563EB", edgecolor="white", alpha=0.8)
ax.axvline(0, color="#DC2626", linestyle="--", linewidth=1.5)
ax.set_xlabel("Residual (min)")
ax.set_ylabel("Frequency")
ax.set_title(f"LightGBM: Residual Distribution (mean={residuals.mean():.2f})", fontweight="bold")

# Panel C: Residuals vs Predicted (LightGBM)
ax = axes[1, 0]
ax.scatter(y_pred_lgbm[sample_idx], residuals[sample_idx],
           alpha=0.05, s=2, color="#2563EB")
ax.axhline(0, color="#DC2626", linestyle="--", linewidth=1.5)
ax.set_xlim(-20, 100)
ax.set_ylim(-100, 100)
ax.set_xlabel("Predicted DepDelay (min)")
ax.set_ylabel("Residual (min)")
ax.set_title("LightGBM: Residuals vs Predicted", fontweight="bold")

# Panel D: RMSE comparison bar chart
ax = axes[1, 1]
colors = ["#94A3B8", "#60A5FA", "#3B82F6", "#DC2626"]
bars = ax.barh(results["Model"], results["RMSE"], color=colors, edgecolor="white")
for bar, val in zip(bars, results["RMSE"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=10)
ax.set_xlabel("RMSE (minutes)")
ax.set_title("Model Comparison — RMSE", fontweight="bold")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(REPORT_DIR / "regression_residual_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 10: Feature importance — LightGBM Regressor

In [ ]:
imp = pd.Series(lgbm_reg.feature_importances_, index=X_train.columns)
top20 = imp.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.sort_values().plot.barh(ax=ax, color="#DC2626", edgecolor="white")
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("Top 20 Features — LightGBM Regressor", fontweight="bold")
for i, (val, name) in enumerate(zip(top20.sort_values(), top20.sort_values().index)):
    ax.text(val + imp.max() * 0.01, i, f"{val:.0f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(REPORT_DIR / "regression_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 11: Linear Regression — Top coefficients

In [ ]:
coefs = pd.Series(lr.coef_, index=X_train.columns)
top_pos = coefs.nlargest(10)
top_neg = coefs.nsmallest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
top_pos.sort_values().plot.barh(ax=ax, color="#DC2626", edgecolor="white")
ax.set_xlabel("Coefficient (standardized)")
ax.set_title("Top 10 Positive Coefficients — Increase Delay", fontweight="bold")

ax = axes[1]
top_neg.sort_values(ascending=False).plot.barh(ax=ax, color="#2563EB", edgecolor="white")
ax.set_xlabel("Coefficient (standardized)")
ax.set_title("Top 10 Negative Coefficients — Decrease Delay", fontweight="bold")

plt.tight_layout()
plt.savefig(REPORT_DIR / "regression_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 12: Summary

In [ ]:
print("=" * 65)
print("REGRESSION MODELING — SUMMARY")
print("=" * 65)
print(f"Target: DepDelay (departure delay in minutes)")
print(f"Train: {len(X_train):,} flights (Jan–Oct 2024)")
print(f"Test:  {len(X_test):,} flights (Nov–Dec 2024)")
print(f"Features: {X_train.shape[1]}")
print()
print(results.to_string(index=False, float_format="{:.4f}".format))
print()
print("Key findings:")
print(f"  1. LightGBM is the best regressor (RMSE={rmse_lgbm:.2f}, R²={r2_lgbm:.4f})")
print(f"  2. Linear/Ridge give similar results — the problem is inherently non-linear")
print(f"  3. prev_flight_arr_delay remains the dominant feature in regression too")
print(f"  4. Delay prediction is hard — even the best model explains ~{r2_lgbm*100:.0f}% of variance")
print("=" * 65)